### Build datasets for prediction

Relies on prediction times file from __*make_prediction_times.ipynb*__.

Make one df that has all the variables we need for all the models (i.e. include num_days, weeks, months, scaleSize, size(scaleSize), TX and CA.

At the end it should look just like the real data I use for modeling

In [1]:
import pandas as pd
import numpy as np

In [2]:
dfPredTimes = pd.read_csv('prediction_times.csv', parse_dates=True, index_col=0)

In [3]:
dfPredTimes.head()

,predDate,predWeek,predMonth,num_days,num_weeks,num_months
0,1998-03-31,1998-03-30/1998-04-05,1998-03,81,12,2
1,1998-06-30,1998-06-29/1998-07-05,1998-06,172,25,5
2,1998-09-30,1998-09-28/1998-10-04,1998-09,264,38,8
3,1998-12-31,1998-12-28/1999-01-03,1998-12,356,51,11
4,1999-03-31,1999-03-29/1999-04-04,1999-03,446,64,14


### What do we need to predict for OLS linear model with time, size, state.
1. num_days - end of every quarter from 1998 through 2018
2. size_kw (1, 2, 3)
3. states (CA, TX)

#### And what else?
1. num_weeks
2. num_months
3. scaleSize

### How to build this up

when it's done it'll look like

    num_days = dfPredTimes.num_days (times 2 states * 3 sizes)

start with dfPredTimes.num_days; make 3 copies and glue together;
fill in sizes 

In [4]:
theTimes = dfPredTimes[['num_days', 'num_weeks', 'num_months']]

In [5]:
thePredFrame = theTimes.append(theTimes)

In [6]:
thePredFrame = thePredFrame.append(theTimes)

In [7]:
len(thePredFrame)

252

In [8]:
len(theTimes)

84

In [9]:
84*3

252

### Okay, I have 3 sets of times; now fill in 3 sizes [3.84, 6.0, 8.5]

In [10]:
theSizes = np.array([[3.84]*84,[6.0]*84,[8.5]*84])

In [11]:
theSizes = theSizes.ravel()

In [12]:
thePredFrame = thePredFrame.assign(size_kw=theSizes)

In [13]:
def showData(df):
    print(df.head())
    print(df.tail())
showData(thePredFrame)

   num_days  num_weeks  num_months  size_kw
0        81         12           2     3.84
1       172         25           5     3.84
2       264         38           8     3.84
3       356         51          11     3.84
4       446         64          14     3.84
    num_days  num_weeks  num_months  size_kw
79      7296       1042         239      8.5
80      7386       1055         242      8.5
81      7477       1068         245      8.5
82      7569       1081         248      8.5
83      7661       1095         251      8.5


#### Reset the index because now it's 0-83, repeated 3 times.

In [14]:
thePredFrame = thePredFrame.reset_index(drop=True)

In [15]:
showData(thePredFrame)

   num_days  num_weeks  num_months  size_kw
0        81         12           2     3.84
1       172         25           5     3.84
2       264         38           8     3.84
3       356         51          11     3.84
4       446         64          14     3.84
     num_days  num_weeks  num_months  size_kw
247      7296       1042         239      8.5
248      7386       1055         242      8.5
249      7477       1068         245      8.5
250      7569       1081         248      8.5
251      7661       1095         251      8.5


#### add in scale size

In [16]:
theScaleSizes = np.array([[1.0]*84,[2.0]*84,[3.0]*84])

In [17]:
theScaleSizes = theScaleSizes.ravel()

In [18]:
thePredFrame = thePredFrame.assign(scaleSize=theScaleSizes)

In [19]:
showData(thePredFrame)

   num_days  num_weeks  num_months  size_kw  scaleSize
0        81         12           2     3.84        1.0
1       172         25           5     3.84        1.0
2       264         38           8     3.84        1.0
3       356         51          11     3.84        1.0
4       446         64          14     3.84        1.0
     num_days  num_weeks  num_months  size_kw  scaleSize
247      7296       1042         239      8.5        3.0
248      7386       1055         242      8.5        3.0
249      7477       1068         245      8.5        3.0
250      7569       1081         248      8.5        3.0
251      7661       1095         251      8.5        3.0


#### Now add the state one-hot columns

In [20]:
theStates = ['state_AZ', 'state_CA', 'state_CT',
       'state_DE', 'state_FL', 'state_MA', 'state_MD', 'state_MN', 'state_NH',
       'state_NJ', 'state_NM', 'state_NV', 'state_NY', 'state_OR', 'state_PA',
       'state_TX', 'state_VT', 'state_WI']

In [21]:
len(theStates)

18

In [22]:
statesData = np.zeros((252, 18))

In [23]:
statesDF = pd.DataFrame(statesData, columns=theStates)

In [24]:
showData(statesDF)

   state_AZ  state_CA  state_CT  state_DE  state_FL  state_MA  state_MD  \
0       0.0       0.0       0.0       0.0       0.0       0.0       0.0   
1       0.0       0.0       0.0       0.0       0.0       0.0       0.0   
2       0.0       0.0       0.0       0.0       0.0       0.0       0.0   
3       0.0       0.0       0.0       0.0       0.0       0.0       0.0   
4       0.0       0.0       0.0       0.0       0.0       0.0       0.0   

   state_MN  state_NH  state_NJ  state_NM  state_NV  state_NY  state_OR  \
0       0.0       0.0       0.0       0.0       0.0       0.0       0.0   
1       0.0       0.0       0.0       0.0       0.0       0.0       0.0   
2       0.0       0.0       0.0       0.0       0.0       0.0       0.0   
3       0.0       0.0       0.0       0.0       0.0       0.0       0.0   
4       0.0       0.0       0.0       0.0       0.0       0.0       0.0   

   state_PA  state_TX  state_VT  state_WI  
0       0.0       0.0       0.0       0.0  
1       0.

#### Glue them together

In [25]:
thePredFrame = pd.concat([thePredFrame, statesDF], axis=1)

In [26]:
showData(thePredFrame)

   num_days  num_weeks  num_months  size_kw  scaleSize  state_AZ  state_CA  \
0        81         12           2     3.84        1.0       0.0       0.0   
1       172         25           5     3.84        1.0       0.0       0.0   
2       264         38           8     3.84        1.0       0.0       0.0   
3       356         51          11     3.84        1.0       0.0       0.0   
4       446         64          14     3.84        1.0       0.0       0.0   

   state_CT  state_DE  state_FL    ...     state_NH  state_NJ  state_NM  \
0       0.0       0.0       0.0    ...          0.0       0.0       0.0   
1       0.0       0.0       0.0    ...          0.0       0.0       0.0   
2       0.0       0.0       0.0    ...          0.0       0.0       0.0   
3       0.0       0.0       0.0    ...          0.0       0.0       0.0   
4       0.0       0.0       0.0    ...          0.0       0.0       0.0   

   state_NV  state_NY  state_OR  state_PA  state_TX  state_VT  state_WI  
0     

In [27]:
thePredFrame.columns.tolist().index('state_CA')

6

In [28]:
thePredFrame.columns.tolist().index('state_TX')

20

In [29]:
len(thePredFrame)

252

### Double the data

In [30]:
thePredFrame = thePredFrame.append(thePredFrame)

In [31]:
showData(thePredFrame)

   num_days  num_weeks  num_months  size_kw  scaleSize  state_AZ  state_CA  \
0        81         12           2     3.84        1.0       0.0       0.0   
1       172         25           5     3.84        1.0       0.0       0.0   
2       264         38           8     3.84        1.0       0.0       0.0   
3       356         51          11     3.84        1.0       0.0       0.0   
4       446         64          14     3.84        1.0       0.0       0.0   

   state_CT  state_DE  state_FL    ...     state_NH  state_NJ  state_NM  \
0       0.0       0.0       0.0    ...          0.0       0.0       0.0   
1       0.0       0.0       0.0    ...          0.0       0.0       0.0   
2       0.0       0.0       0.0    ...          0.0       0.0       0.0   
3       0.0       0.0       0.0    ...          0.0       0.0       0.0   
4       0.0       0.0       0.0    ...          0.0       0.0       0.0   

   state_NV  state_NY  state_OR  state_PA  state_TX  state_VT  state_WI  
0     

### Reset the index because now it's 0-251, repeated twice.

In [32]:
thePredFrame = thePredFrame.reset_index(drop=True)

In [33]:
showData(thePredFrame)

   num_days  num_weeks  num_months  size_kw  scaleSize  state_AZ  state_CA  \
0        81         12           2     3.84        1.0       0.0       0.0   
1       172         25           5     3.84        1.0       0.0       0.0   
2       264         38           8     3.84        1.0       0.0       0.0   
3       356         51          11     3.84        1.0       0.0       0.0   
4       446         64          14     3.84        1.0       0.0       0.0   

   state_CT  state_DE  state_FL    ...     state_NH  state_NJ  state_NM  \
0       0.0       0.0       0.0    ...          0.0       0.0       0.0   
1       0.0       0.0       0.0    ...          0.0       0.0       0.0   
2       0.0       0.0       0.0    ...          0.0       0.0       0.0   
3       0.0       0.0       0.0    ...          0.0       0.0       0.0   
4       0.0       0.0       0.0    ...          0.0       0.0       0.0   

   state_NV  state_NY  state_OR  state_PA  state_TX  state_VT  state_WI  
0     

### set state_CA in first 252 rows

In [34]:
thePredFrame.loc[0:251, 'state_CA'] = 1.0

In [35]:
thePredFrame.loc[252:, 'state_TX'] = 1.0

In [36]:
showData(thePredFrame)

   num_days  num_weeks  num_months  size_kw  scaleSize  state_AZ  state_CA  \
0        81         12           2     3.84        1.0       0.0       1.0   
1       172         25           5     3.84        1.0       0.0       1.0   
2       264         38           8     3.84        1.0       0.0       1.0   
3       356         51          11     3.84        1.0       0.0       1.0   
4       446         64          14     3.84        1.0       0.0       1.0   

   state_CT  state_DE  state_FL    ...     state_NH  state_NJ  state_NM  \
0       0.0       0.0       0.0    ...          0.0       0.0       0.0   
1       0.0       0.0       0.0    ...          0.0       0.0       0.0   
2       0.0       0.0       0.0    ...          0.0       0.0       0.0   
3       0.0       0.0       0.0    ...          0.0       0.0       0.0   
4       0.0       0.0       0.0    ...          0.0       0.0       0.0   

   state_NV  state_NY  state_OR  state_PA  state_TX  state_VT  state_WI  
0     

#### Check the middle

In [37]:
thePredFrame.iloc[250:260, :]

,num_days,num_weeks,num_months,size_kw,scaleSize,state_AZ,state_CA,state_CT,state_DE,state_FL,...,state_NH,state_NJ,state_NM,state_NV,state_NY,state_OR,state_PA,state_TX,state_VT,state_WI
250,7569,1081,248,8.50,3.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
251,7661,1095,251,8.50,3.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
252,81,12,2,3.84,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
253,172,25,5,3.84,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
254,264,38,8,3.84,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
255,356,51,11,3.84,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
256,446,64,14,3.84,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
257,537,77,17,3.84,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
258,629,90,20,3.84,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
259,721,103,23,3.84,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


#### lookin' good; save it.

In [38]:
thePredFrame.to_csv('mod_pred_data.csv')

#### make sure.  okay.

In [39]:
thing = pd.read_csv('mod_pred_data.csv', index_col=0)

In [40]:
thing.iloc[250:260, :]

,num_days,num_weeks,num_months,size_kw,scaleSize,state_AZ,state_CA,state_CT,state_DE,state_FL,...,state_NH,state_NJ,state_NM,state_NV,state_NY,state_OR,state_PA,state_TX,state_VT,state_WI
250,7569,1081,248,8.50,3.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
251,7661,1095,251,8.50,3.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
252,81,12,2,3.84,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
253,172,25,5,3.84,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
254,264,38,8,3.84,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
255,356,51,11,3.84,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
256,446,64,14,3.84,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
257,537,77,17,3.84,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
258,629,90,20,3.84,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
259,721,103,23,3.84,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [41]:
showData(thing)

   num_days  num_weeks  num_months  size_kw  scaleSize  state_AZ  state_CA  \
0        81         12           2     3.84        1.0       0.0       1.0   
1       172         25           5     3.84        1.0       0.0       1.0   
2       264         38           8     3.84        1.0       0.0       1.0   
3       356         51          11     3.84        1.0       0.0       1.0   
4       446         64          14     3.84        1.0       0.0       1.0   

   state_CT  state_DE  state_FL    ...     state_NH  state_NJ  state_NM  \
0       0.0       0.0       0.0    ...          0.0       0.0       0.0   
1       0.0       0.0       0.0    ...          0.0       0.0       0.0   
2       0.0       0.0       0.0    ...          0.0       0.0       0.0   
3       0.0       0.0       0.0    ...          0.0       0.0       0.0   
4       0.0       0.0       0.0    ...          0.0       0.0       0.0   

   state_NV  state_NY  state_OR  state_PA  state_TX  state_VT  state_WI  
0     